In [1]:
from dataset import *
from evaluate import *
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


 
- Arrowhead Beach: 36.236102, -76.695387 (Planet 4)

- Cannon's Ferry: 36.271557, -76.672967 (Planet 6)

- Rocky Hock: 36.183649, -76.719210 (Planet 3)

- Chowan River Bridge: 36.055233, -76.686425 (Planet 5)

- Point Comfort: 36.169458, -76.751827 (Planet 1)

- Mt. Gould : 36.123866, -76.742630 (Planet 2)

In [2]:
import os
import torch
import numpy as np
import rasterio as rio
from rasterio.windows import Window
from pyproj import Transformer

class SentinelDataset:
    def __init__(self, lat, lon):
        """
        PyTorch dataset for extracting Sentinel data around specific lat/lon coordinates.

        Args:
            lat (float): Latitude of the crop center.
            lon (float): Longitude of the crop center.
        """
        self.lat = lat
        self.lon = lon

        # Directory containing the label files
        lbl_dir = '/work/srs108/all_data'
        self.cicyano_paths = []

        # Gather valid .tif file paths and filter invalid ones
        for folder in tqdm(sorted(os.listdir(lbl_dir))):
            print(folder)
            if folder not in [".DS_Store", ".ipynb_checkpoints"]:
                folder_path = os.path.join(lbl_dir, folder)
                for file in sorted(os.listdir(folder_path)):
                    if file.endswith('.tif'):
                        parts = file.split('.')
                        name = parts[-3]
                        if name == 'CIcyano':
                            file_path = os.path.join(folder_path, file)
                            self.cicyano_paths.append(file_path)
  

    def __len__(self):
        return len(self.cicyano_paths)

    def __getitem__(self, idx):

        file = self.cicyano_paths[idx]

        with rio.open(file) as rds:
            data = rds.read([1])

            crop_size = 3  # Crop size in pixels

            # Convert lat/lon to image coordinates
            transformer = Transformer.from_crs("EPSG:4326", rds.crs, always_xy=True)
            xx, yy = transformer.transform(self.lon, self.lat)
            row, col = rds.index(xx, yy)

            # Define the crop window
            row_start = max(row - crop_size, 0)
            row_end = min(row + crop_size, rds.height)
            col_start = max(col - crop_size, 0)
            col_end = min(col + crop_size, rds.width)

            window = Window.from_slices((row_start, row_end), (col_start, col_end))
            data = rds.read(window=window)
            lbl_arr = np.array(data)

            fpt_avg = np.where((31 <= lbl_arr) & (lbl_arr<= 249), lbl_arr, 0) #bloom pixel avg only
            fpt_avg = fpt_avg[fpt_avg !=0]
            mean_value = fpt_avg.mean() if fpt_avg.size > 0 else 0

            #label with land (2), cloud (3), bloom (1), else (0)
            fptdata = np.where(lbl_arr == 252, 2, np.where(lbl_arr == 253, 3, np.where((31 <= lbl_arr) & (lbl_arr<= 249), 1, 0)))

            if 31 <= mean_value <= 249:
                class_lbl = 1
            else: class_lbl = 0
            sentinel = torch.tensor(data, dtype=torch.float32)

            dy = file.split('/')[-1].split('.')[1][:4]
            dm = file.split('/')[-1].split('.')[2][:2]
            dd = file.split('/')[-1].split('.')[2][2:]
            d = dy +'-'+ dm +'-'+ dd
            sample = {
                'sentinel': sentinel,
                'pred': class_lbl,
                'date': str(d)
            }
        return sample


In [3]:
test = SentinelDataset(36.055382, -76.610319)
batch_size = 16

chowan_loader = DataLoader(test, batch_size=batch_size, shuffle = False)


 45%|████▌     | 5/11 [00:00<00:00, 44.55it/s]

.DS_Store
.ipynb_checkpoints
albemarle_2016_archive
albemarle_2017_archive
albemarle_2018_archive
albemarle_2019_archive
albemarle_2020_archive


100%|██████████| 11/11 [00:00<00:00, 25.63it/s]

albemarle_2021_archive
albemarle_2022_archive
albemarle_2023_archive
albemarle_2024_archive


In [5]:
import datetime
from tqdm import tqdm

tracking = {'date': [], 'pred': []} 

with torch.no_grad():
    for i, batch in tqdm(enumerate(chowan_loader)):
        # Move sentinel data to the device
        y = batch['pred']
        c = batch['date'] 

#         output = model(y)
#         output_binary = (output.cpu().detach().numpy() >= 0).astype(int)  # Convert to binary predictions
        
#         torch.cuda.empty_cache()  # Clear CUDA cache
        
        dates = [
            datetime.datetime.strptime(fp, "%Y-%m-%d").strftime("%Y-%m-%d") 
            for fp in c
        ]
        
        # Update the tracking dictionary
        tracking['date'].extend(dates)  # Append formatted dates
        tracking['pred'].extend(y.tolist())  # Append predictions

# `tracking` now contains the 'date' and 'pred' for all batches


217it [04:36,  1.28s/it]


In [6]:
df = pd.DataFrame(tracking)

df.iloc[10:20]

,date,pred
10,2016-05-07,1
11,2016-05-09,0
12,2016-05-10,0
13,2016-05-11,0
14,2016-05-13,0
15,2016-05-14,1
16,2016-05-15,1
17,2016-05-17,0
18,2016-05-18,0
19,2016-05-19,0


In [7]:
df.to_csv('/hpc/home/srs108/abemarle/planet/self/sent.csv', index=False)